# NB_lab_test_trends (Silver → Gold)

**Target table:** `adb_caresync.gold.lab_test_trends`  
**Grain:** one row per test type (`test_name`) per calendar month  
**Load type:** full rebuild every run (`CREATE OR REPLACE TABLE ... AS SELECT`)

**How it works**
1. **monthly** – group `silver.lab_results` by `test_name`, year and month, and count total / abnormal / critical tests.
2. **monthly_rates** – calculate the abnormal and critical rates (%).
3. **monthly_with_trend** – use `LAG()` to get the previous month's abnormal rate for the same test.
4. **Final select** – `LEFT JOIN` `silver.calendar` on the first day of the month to get `month_name` and `quarter_name`, and calculate the month-over-month change.

**Notes**
* `abnormal_rate_change_pct` is a change in **percentage points** (e.g. 12.5% → 15.0% = `2.50`).
* It is `NULL` for a test's first month because there is no earlier month to compare with.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresync.gold.lab_test_trends
COMMENT 'Gold: one row per test type per month with volumes, rates and month-over-month trend'
AS
-- Step 1: monthly counts per test type
WITH monthly AS (
  SELECT
    test_name,
    year(test_date)                       AS year,
    month(test_date)                      AS month,
    COUNT(*)                              AS total_tests,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count
  FROM adb_caresync.silver.lab_results
  GROUP BY test_name, year(test_date), month(test_date)
),

-- Step 2: rates in %
monthly_rates AS (
  SELECT
    *,
    CAST(abnormal_test_count / total_tests * 100 AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(critical_test_count / total_tests * 100 AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly
),

-- Step 3: previous month's abnormal rate for the same test
monthly_with_trend AS (
  SELECT
    *,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM monthly_rates
)

-- Step 4: add calendar names and the month-over-month change
SELECT
  m.test_name,
  m.year,
  m.month,
  c.month_name,
  c.quarter_name,
  CAST(m.total_tests         AS INT) AS total_tests,
  CAST(m.abnormal_test_count AS INT) AS abnormal_test_count,
  CAST(m.critical_test_count AS INT) AS critical_test_count,
  m.abnormal_rate_pct,
  m.critical_rate_pct,
  m.prev_month_abnormal_rate_pct,
  CAST(m.abnormal_rate_pct - m.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct  -- NULL when no prior month
FROM monthly_with_trend m
LEFT JOIN adb_caresync.silver.calendar c
  ON c.date = make_date(m.year, m.month, 1);   -- first day of the month (same as date_trunc('month', test_date))

In [0]:
# %sql
# -- Each (test_name, year, month) should appear once, and the total tests should match silver.lab_results
# SELECT
#   COUNT(*)                                              AS gold_rows,
#   COUNT(DISTINCT test_name, year, month)                AS distinct_test_months,
#   SUM(total_tests)                                      AS total_tests,
#   (SELECT COUNT(*) FROM adb_caresync.silver.lab_results) AS silver_lab_rows,
#   COUNT_IF(month_name IS NULL)                          AS missing_calendar_rows
# FROM adb_caresync.gold.lab_test_trends;

In [0]:
# %sql
# SELECT * FROM adb_caresync.gold.lab_test_trends ORDER BY test_name, year, month LIMIT 20;

In [0]:
# The table is built in a %sql cell, so there is no Python DataFrame.
# Read the gold table back into a DataFrame to get the row count.
updated_df = spark.table("adb_caresync.gold.lab_test_trends")
record_count = updated_df.count()


dbutils.notebook.exit(str(record_count))